In [90]:
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

In [91]:
path = "../data/raw/"

all_csv = os.listdir(path)
df_list = []
for file in all_csv:
    path_file = os.path.join(path,file)
    df_list.append(pd.read_csv(path_file))

df = pd.concat(df_list)
df.sort_values(by='respondent_id',inplace=True,ignore_index=True)
df.index += 1
df.head()

,age_group,gender,college_year,monthly_budget_range,monthly_budget_exact,income_source,primary_upi_app,weekly_tx_range,weekly_tx_count,pct_unplanned,...,regret_other_text,post_regret_action,regret_intensity,hidden_purchase,tracks_expenses,ran_out_of_money,perceives_upi_risky,upi_usage_reason,regret_description,respondent_id
1,22-25,Male,Ug,₹6000+ (Force fully leaving in the college hos...,18000.0,100% Parents Money,PhonePe,20+ (You are the reason companies are profitab...,36.0,20-40% (Understandable),...,NaN,Accept it or move on (aaj tak nahi huva),3,No (we don't believe),"No, I need some surprise in life","Yes, regularly",Yes,I feel comfortable using UPI,Marrow course,1000
2,18-21,Male,Ug,₹1000 - ₹3000 (Mess food is not enough for me),3500.0,100% Parents Money,PhonePe,20+ (You are the reason companies are profitab...,40.0,20-40% (Understandable),...,NaN,Accept it or move on (aaj tak nahi huva),3,No (we don't believe),"No, I need some surprise in life","Yes, Occasionally",Yes,I feel comfortable using UPI,NaN,1001
3,18-21,Male,Ug,₹1000 - ₹3000 (Mess food is not enough for me),2000.0,100% Parents Money,PhonePe,20+ (You are the reason companies are profitab...,21.0,40-60% (Mostly it on the food),...,NaN,Accept it or move on (aaj tak nahi huva),1,Maybe (don't want to accept it),"I try, but give up by one or two week (Relatable)",Rarely,Yes,I feel comfortable using UPI,NaN,1002
4,18-21,Male,Ug,₹3000 - ₹6000 (I need to go out of this campus),NaN,100% Parents Money,Google Pay (GPay),15-20 (Loves eating too much),NaN,60-80% (Probably you love everything you see o...,...,NaN,Accept it or move on (aaj tak nahi huva),1,Maybe (don't want to accept it),"No, I need some surprise in life",Rarely,Yes,I feel comfortable using UPI,NaN,1003
5,Under 18,Female,Ug,₹3000 - ₹6000 (I need to go out of this campus),NaN,100% Parents Money,PhonePe,8-15 (The apps know your face by now),NaN,20-40% (Understandable),...,NaN,Accept it or move on (aaj tak nahi huva),2,NaN,"I try, but give up by one or two week (Relatable)","Yes, Occasionally",Yes,I feel comfortable using UPI,NaN,1004


In [92]:
df_clean = df.iloc[:,[31,0,1,2]]

In [93]:
df['monthly_budget_range'].unique()

array(['₹6000+ (Force fully leaving in the college hostel)',
       '₹1000 - ₹3000 (Mess food is not enough for me)',
       '₹3000 - ₹6000 (I need to go out of this campus)',
       'Less than ₹1000 (Enjoy hostel Wi-Fi and mess food)',
       '₹6000+ (Force fully living in the college hostel)'], dtype=object)

In [94]:
df_clean['monthly_budget_range'] = df['monthly_budget_range'].replace({
    '₹6000+ (Force fully leaving in the college hostel)' : 'Rs6000+',
       '₹1000 - ₹3000 (Mess food is not enough for me)' : 'Rs1000 - Rs3000',
       '₹3000 - ₹6000 (I need to go out of this campus)' : 'Rs3000 - Rs6000',
       'Less than ₹1000 (Enjoy hostel Wi-Fi and mess food)' : '< Rs1000',
       '₹6000+ (Force fully living in the college hostel)' : 'Rs6000+'
})

income_map = {
    'Rs6000+' : 7000,
    'Rs1000 - Rs3000' : 2000,
    'Rs3000 - Rs6000' : 4500,
    '< Rs1000' : 750
}

df_clean['avg_monthly_budget'] = df['monthly_budget_exact'].fillna(df_clean['monthly_budget_range'].map(income_map))

In [95]:
df['income_source'].value_counts()

income_source
100% Parents Money                                  86
Mix of parents + I earn something on the side       10
I have an internship / part-time job                 5
I run a side hustle / freelance / small business     4
Name: count, dtype: int64

In [96]:
df_clean['income_source'] = df['income_source']

In [97]:
df['primary_upi_app'].value_counts()

primary_upi_app
Google Pay (GPay)               47
PhonePe                         28
Paytm                           14
Bhim UPI                         5
Navi UPI                         5
Cred                             2
Amazon Pay                       1
Google pay as well as Paytm      1
PayZapp                          1
Fampay                           1
Name: count, dtype: int64

In [98]:
df_clean['primary_upi_app'] = df['primary_upi_app'].replace({'Google pay as well as Paytm ' : 'Paytm'})
df_clean['primary_upi_app'].value_counts()

primary_upi_app
Google Pay (GPay)    47
PhonePe              28
Paytm                15
Bhim UPI              5
Navi UPI              5
Cred                  2
Amazon Pay            1
PayZapp               1
Fampay                1
Name: count, dtype: int64

In [99]:
df['weekly_tx_range'].value_counts()

weekly_tx_range
4-7 (Okay this is reasonable )                        40
8-15 (The apps know your face by now)                 29
1-3 (No comments)                                     16
15+ (You are the reason companies are profitable )    16
20+ (You are the reason companies are profitable )     3
15-20 (Loves eating too much)                          1
Name: count, dtype: int64

In [100]:
df_clean['weekly_tx_range'] = df['weekly_tx_range'].replace({
    '1-3 (No comments)' : '1-3',
    '4-7 (Okay this is reasonable )' : '4-7',
    '8-15 (The apps know your face by now)' : '8-15',
    '15+ (You are the reason companies are profitable )' : '15+',
    '20+ (You are the reason companies are profitable )' : '15+',
    '15-20 (Loves eating too much)' : '15+'
})

upi_tx_map = {
    '1-3' : 2,
    '4-7' : 5,
    '8-15': 12,
    '15+': 18
}

df_clean['avg_weekly_tx'] = df['weekly_tx_count'].fillna(df_clean['weekly_tx_range'].map(upi_tx_map))